# Robust Invoice Parsing Lab
Use this notebook with one unfamiliar PDF at a time. It first sends the PDF with high visual detail. If that fails, it extracts selectable text locally and retries. Finally, you can correct any field before saving a CSV.

## 1. Setup
Run this once after activating the project environment.

In [ ]:
%pip install -r requirements.txt

## 2. Choose one PDF
Copy your PDF into this project, then change `PDF_PATH`. Do not paste your API key into the notebook. It reads `OPENAI_API_KEY` from your environment and securely prompts if it is missing.

In [ ]:
import os
from getpass import getpass
from pathlib import Path
import pandas as pd
from pypdf import PdfReader
from invoice_helper import extract_invoice_robust, invoice_to_row

PDF_PATH = Path('samples/storereceipt.pdf')  # change this
MODEL = os.getenv('OPENAI_MODEL', 'gpt-5.6')
API_KEY = os.getenv('OPENAI_API_KEY') or getpass('OpenAI API key: ')

assert PDF_PATH.exists(), f'File not found: {PDF_PATH}'
assert PDF_PATH.suffix.lower() == '.pdf', 'Please select a PDF file'
pdf_bytes = PDF_PATH.read_bytes()
print(f'File: {PDF_PATH.name}')
print(f'Size: {len(pdf_bytes) / 1024:.1f} KB')
print(f'Model: {MODEL}')

## 3. Inspect local PDF text
This diagnostic does not call the API. Empty output usually means the invoice is scanned; the visual PDF method can still work.

In [ ]:
reader = PdfReader(PDF_PATH)
local_text = '\n'.join(page.extract_text() or '' for page in reader.pages)
print(f'Pages: {len(reader.pages)}')
print(f'Extracted characters: {len(local_text.strip())}')
print('\nPreview:\n', local_text[:1500] or '[No selectable text]')

## 4. Run robust extraction
This makes one API call. A second call occurs only if the direct PDF attempt fails and readable local text exists.

In [ ]:
try:
    invoice, method, warnings = extract_invoice_robust(
        pdf_bytes, PDF_PATH.name, API_KEY, MODEL
    )
    print('Successful method:', method)
    for warning in warnings:
        print('Warning:', warning)
    display(pd.DataFrame([invoice_to_row(invoice, PDF_PATH.name)]))
except Exception as error:
    invoice = None
    print(type(error).__name__)
    print(error)

## 5. Review and correct
AI extraction can be wrong. Edit any value below after comparing it with the PDF.

In [ ]:
if invoice is None:
    corrected = {
        'date': None, 'vendor': None, 'invoice_number': None,
        'amount': None, 'currency': None,
        'original_filename': PDF_PATH.name, 'suggested_filename': None,
        'notes': 'Manual entry required'
    }
else:
    corrected = invoice_to_row(invoice, PDF_PATH.name)

# Example correction: corrected['vendor'] = 'Correct Vendor Name'
corrected

## 6. Export the reviewed row

In [ ]:
output = pd.DataFrame([corrected])
output.to_csv('my_invoice_result.csv', index=False)
print('Saved: my_invoice_result.csv')
output